# LeetCode 3794 — Reverse String Prefix bằng Python

Link đề: https://leetcode.com/problems/reverse-string-prefix/ · Độ khó: **Easy** · Chủ đề: String, Two Pointers

Skeleton LeetCode:

```python
class Solution:
    def reversePrefix(self, s: str, k: int) -> str:
        ...
```

> Ghi chú: tên bài, đề, ví dụ, constraints và chữ ký hàm đã được đối chiếu qua WebSearch / leetcode.doocs.org (một nguồn mirror ghi nhầm tên khác cho ID này; trang doocs cho đúng đề "Reverse String Prefix").

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đảo ngược `k` ký tự **đầu tiên** của chuỗi `s`, giữ nguyên phần còn lại, rồi trả về chuỗi mới.
2. Nắm vững **slicing** chuỗi: `s[:k]`, `s[k:]`, `s[::-1]` và vì sao chuỗi Python là **bất biến**.
3. Biết cách đảo một đoạn bằng **hai con trỏ** trên list ký tự và bằng **ngăn xếp** (stack).
4. Viết 5 cách khác nhau (slicing, hai con trỏ, ngăn xếp, `reversed` + `join`, đệ quy) và đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: gán `s[i] = ...` trực tiếp, nhầm `s[:k:-1]`, đảo cả chuỗi, và quên nối phần đuôi `s[k:]`.

## 2. Đề bài & ví dụ

Cho chuỗi `s` và số nguyên `k`. Hãy **đảo ngược `k` ký tự đầu tiên** của `s` và trả về chuỗi kết quả. Các ký tự từ vị trí `k` trở đi giữ nguyên.

| Ví dụ | s, k | Output | Giải thích |
|---|---|---|---|
| 1 | `"abcd"`, `2` | `"bacd"` | `"ab"` đảo thành `"ba"`, phần còn lại `"cd"` giữ nguyên |
| 2 | `"xyz"`, `3` | `"zyx"` | Đảo cả chuỗi vì `k = len(s)` |
| 3 | `"hey"`, `1` | `"hey"` | Đảo một ký tự thì không đổi |

Constraints: `1 <= s.length <= 100`, `s` chỉ gồm chữ cái tiếng Anh thường, `1 <= k <= s.length`.

In [1]:
for s, k in [("abcd", 2), ("xyz", 3), ("hey", 1)]:
    head, tail = s[:k], s[k:]
    print(f"s={s!r} k={k}  đầu={head!r} -> đảo={head[::-1]!r}  đuôi={tail!r}  kết quả={head[::-1] + tail!r}")

s='abcd' k=2  đầu='ab' -> đảo='ba'  đuôi='cd'  kết quả='bacd'
s='xyz' k=3  đầu='xyz' -> đảo='zyx'  đuôi=''  kết quả='zyx'
s='hey' k=1  đầu='h' -> đảo='h'  đuôi='ey'  kết quả='hey'


## 3. Kiến thức nền

### 3.1. Chỉ số và slicing chuỗi

Chuỗi Python là dãy ký tự đánh chỉ số từ `0`. `s[a:b]` lấy các ký tự từ chỉ số `a` đến `b-1` (nửa mở, **không** gồm `b`). Bỏ `a` thì mặc định `0`; bỏ `b` thì mặc định đến hết chuỗi. Vì vậy `s[:k]` là `k` ký tự đầu và `s[k:]` là phần còn lại, và hai phần ghép lại đúng bằng `s`.

In [2]:
s = "abcdef"
print(s[:2], s[2:], s[:2] + s[2:] == s)
print(s[1:4], s[:0] == "", s[6:] == "")

ab cdef True
bcd True True


### 3.2. Đảo chuỗi bằng `[::-1]`

`s[::-1]` là slicing với bước `-1`: đi từ cuối về đầu, cho chuỗi đảo ngược. Cú pháp tổng quát `s[start:stop:step]`. Lưu ý: kết hợp bước âm với `start/stop` rất dễ nhầm (xem mục lỗi).

In [3]:
s = "abcd"
print(s[::-1], s[:2][::-1], s[1::-1])
print(s[::2], s[::-2])

dcba ba ba
ac db


### 3.3. Chuỗi là bất biến (immutable)

Không thể sửa một ký tự bằng `s[i] = 'x'` (gây `TypeError`). Muốn "sửa", ta tạo **chuỗi mới** hoặc chuyển sang `list` ký tự, sửa, rồi `''.join(list)`.

In [4]:
s = "abcd"
try:
    s[0] = "z"
except TypeError as e:
    print("TypeError:", e)
chars = list(s)
chars[0] = "z"
print(chars, ''.join(chars), "(s vẫn là", s, ")")

TypeError: 'str' object does not support item assignment
['z', 'b', 'c', 'd'] zbcd (s vẫn là abcd )


### 3.4. `list`, hoán đổi và hai con trỏ

Để đảo một đoạn trong list: dùng hai chỉ số `i = 0`, `j = k - 1`, hoán đổi `chars[i], chars[j] = chars[j], chars[i]` rồi `i += 1; j -= 1` đến khi `i >= j`. Phép gán song song của Python hoán đổi không cần biến tạm.

In [5]:
chars = list("abcd")
i, j = 0, 1
while i < j:
    chars[i], chars[j] = chars[j], chars[i]
    i += 1
    j -= 1
print(''.join(chars))
a, b = 1, 2
a, b = b, a
print(a, b)

bacd
2 1


### 3.5. `reversed`, `join` và ngăn xếp

`reversed(x)` trả iterator đi từ cuối về đầu. `''.join(iterable_of_str)` nối các chuỗi. **Ngăn xếp** (stack) vào sau ra trước: `push` bằng `append`, `pop` lấy phần tử cuối; đẩy `k` ký tự vào rồi pop hết sẽ cho thứ tự đảo.

In [6]:
s = "abcd"
print(''.join(reversed(s[:2])))
stack = []
for c in "ab":
    stack.append(c)
out = ''
while stack:
    out += stack.pop()
print(out)

ba
ba


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Hãy tưởng tượng một hàng chữ cái. Bạn cầm **`k` chữ đầu tiên**, lật ngược thứ tự của chúng, rồi đặt lại vào đúng chỗ; các chữ phía sau không đụng tới. Vậy kết quả = `(k ký tự đầu đã đảo) + (phần còn lại)`.

Tính tay `s = "abcd"`, `k = 2`: đầu `"ab"`, đảo thành `"ba"`, đuôi `"cd"` -> `"bacd"`.

Hai trường hợp biên đáng chú ý:

- `k = 1`: đoạn chỉ có một ký tự, đảo không đổi -> kết quả `s`.
- `k = len(s)`: đảo cả chuỗi -> `s[::-1]`.

### Bảng tính tay

| s, k | đầu `s[:k]` | đầu đã đảo | đuôi `s[k:]` | kết quả |
|---|---|---|---|---|
| `"abcd"`, 2 | `ab` | `ba` | `cd` | `bacd` |
| `"xyz"`, 3 | `xyz` | `zyx` | (rỗng) | `zyx` |
| `"hey"`, 1 | `h` | `h` | `ey` | `hey` |
| `"abcdef"`, 4 | `abcd` | `dcba` | `ef` | `dcbaef` |

In [7]:
for s, k in [("abcd", 2), ("xyz", 3), ("hey", 1), ("abcdef", 4)]:
    print(s, k, s[:k], s[:k][::-1], s[k:] or "(rỗng)", "->", s[:k][::-1] + s[k:])

abcd 2 ab ba cd -> bacd
xyz 3 xyz zyx (rỗng) -> zyx
hey 1 h h ey -> hey
abcdef 4 abcd dcba ef -> dcbaef


## 5. Cách 1 — Brute Force / baseline

Dựng kết quả **từng ký tự một** bằng vòng lặp: với vị trí `i < k` lấy ký tự `s[k - 1 - i]` (ký tự đối xứng trong đoạn đầu); với `i >= k` lấy `s[i]`. Không dùng slicing đảo, nên thấy rõ ánh xạ vị trí.

Nút thắt: nối chuỗi bằng `+=` trong vòng lặp có thể tạo nhiều chuỗi trung gian; với độ dài `<= 100` không đáng kể, nhưng `join` hoặc slicing gọn và nhanh hơn.

In [8]:
class SolutionLoop:
    def reversePrefix(self, s, k):
        res = ""
        for i in range(len(s)):
            if i < k:
                res += s[k - 1 - i]        # vị trí i trong đoạn đầu lấy ký tự đối xứng
            else:
                res += s[i]
        return res

print(SolutionLoop().reversePrefix("abcd", 2), SolutionLoop().reversePrefix("xyz", 3), SolutionLoop().reversePrefix("hey", 1))

bacd zyx hey


**Độ phức tạp:** Time `O(n)` (đến `O(n²)` nếu tính cả chi phí sao chép khi `+=` trên chuỗi dài), Space `O(n)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** tách chuỗi thành hai phần độc lập `s[:k]` và `s[k:]`, chỉ phần đầu cần đảo; ghép lại bằng `+`.

**Quy tắc "nếu… thì…":**

- Nếu vị trí `i < k` -> ký tự kết quả là `s[k - 1 - i]`.
- Nếu `i >= k` -> ký tự kết quả là `s[i]` (không đổi).
- Nếu `k = 1` -> kết quả `s`; nếu `k = n` -> kết quả `s[::-1]`.
- Độ dài kết quả luôn bằng `len(s)` và là **hoán vị** các ký tự của `s`.

**Invariant (hai con trỏ):** tại mọi thời điểm, các ký tự ở chỉ số `< i` và `> j` trong đoạn `[0, k-1]` đã ở đúng vị trí đảo; đoạn `[i, j]` chưa xử lý.

In [9]:
# Kiểm chứng: slicing = ánh xạ vị trí s[k-1-i]; kết quả là hoán vị của s
import random
random.seed(3794)
for _ in range(1500):
    n = random.randint(1, 30)
    s = ''.join(random.choice('abc') for _ in range(n))
    k = random.randint(1, n)
    r = s[:k][::-1] + s[k:]
    assert r == ''.join(s[k - 1 - i] if i < k else s[i] for i in range(n))
    assert len(r) == n and sorted(r) == sorted(s) and r[k:] == s[k:]
print("slicing khớp ánh xạ vị trí, và kết quả là hoán vị của s")

slicing khớp ánh xạ vị trí, và kết quả là hoán vị của s


## 7. Các cách tiếp theo

### Cách 2 — Slicing

`s[:k][::-1] + s[k:]`: một dòng, nhanh vì chạy ở tầng C của Python. Đây là lời giải thông dụng nhất.

In [10]:
class SolutionSlice:
    def reversePrefix(self, s, k):
        return s[:k][::-1] + s[k:]

print(SolutionSlice().reversePrefix("abcd", 2), SolutionSlice().reversePrefix("xyz", 3), SolutionSlice().reversePrefix("hey", 1))

bacd zyx hey


**Độ phức tạp:** Time `O(n)`, Space `O(n)`.

### Cách 3 — Hai con trỏ trên list

Chuyển `s` thành list ký tự; `i = 0`, `j = k - 1`; hoán đổi và thu hẹp đến khi gặp nhau; cuối cùng `join`. Đây là mẫu "đảo tại chỗ" dùng khắp nơi (mảng, danh sách liên kết).

In [11]:
class SolutionTwoPointers:
    def reversePrefix(self, s, k):
        chars = list(s)
        i, j = 0, k - 1
        while i < j:
            chars[i], chars[j] = chars[j], chars[i]
            i += 1
            j -= 1
        return ''.join(chars)

print(SolutionTwoPointers().reversePrefix("abcd", 2), SolutionTwoPointers().reversePrefix("xyz", 3), SolutionTwoPointers().reversePrefix("hey", 1))

bacd zyx hey


**Độ phức tạp:** Time `O(n)`, Space `O(n)` (list ký tự); phần đảo chỉ `O(k)` hoán đổi.

### Cách 4 — Ngăn xếp

Đẩy `k` ký tự đầu vào stack, rồi `pop` ra theo thứ tự ngược, cuối cùng nối phần đuôi. Hữu ích khi bạn muốn liên hệ với các bài dùng stack.

In [12]:
class SolutionStack:
    def reversePrefix(self, s, k):
        stack = []
        for c in s[:k]:
            stack.append(c)
        out = []
        while stack:
            out.append(stack.pop())
        out.append(s[k:])
        return ''.join(out)

print(SolutionStack().reversePrefix("abcd", 2), SolutionStack().reversePrefix("xyz", 3), SolutionStack().reversePrefix("hey", 1))

bacd zyx hey


**Độ phức tạp:** Time `O(n)`, Space `O(n)`.

### Cách 5 — Đệ quy

Định nghĩa `rev(t)` đảo chuỗi `t` bằng `rev(t[1:]) + t[0]`; kết quả `rev(s[:k]) + s[k:]`. Minh họa cấu trúc bài toán con, nhưng với `k` lớn đệ quy sâu `k` mức (ở đây tối đa 100, an toàn).

In [13]:
class SolutionRecursive:
    def _rev(self, t):
        if len(t) <= 1:
            return t
        return self._rev(t[1:]) + t[0]

    def reversePrefix(self, s, k):
        return self._rev(s[:k]) + s[k:]

print(SolutionRecursive().reversePrefix("abcd", 2), SolutionRecursive().reversePrefix("xyz", 3), SolutionRecursive().reversePrefix("hey", 1))

bacd zyx hey


**Độ phức tạp:** Time `O(k²)` (mỗi tầng cắt chuỗi `O(k)`), Space `O(k)` ngăn xếp; với `k <= 100` vẫn tức thì.

## 8. TRACE từng bước

Trace Cách 3 (hai con trỏ) với `s = "abcdef"`, `k = 4`:

| bước | `i` | `j` | hoán đổi | list sau bước |
|---|---|---|---|---|
| bắt đầu | 0 | 3 | | `a b c d e f` |
| 1 | 0 | 3 | `a` <-> `d` | `d b c a e f` |
| 2 | 1 | 2 | `b` <-> `c` | `d c b a e f` |
| dừng | 2 | 1 | `i >= j` | `d c b a e f` |

Kết quả `"dcbaef"`.

In [14]:
def trace_two_pointers(s, k):
    chars = list(s)
    i, j = 0, k - 1
    print("s =", s, " k =", k, " list:", chars)
    step = 0
    while i < j:
        step += 1
        chars[i], chars[j] = chars[j], chars[i]
        print(f"  bước {step}: hoán đổi chỉ số {i} và {j} -> {''.join(chars)}")
        i += 1
        j -= 1
    print(f"  dừng với i={i}, j={j}")
    return ''.join(chars)

assert trace_two_pointers("abcdef", 4) == "dcbaef"
assert trace_two_pointers("xyz", 3) == "zyx"
assert trace_two_pointers("hey", 1) == "hey"

s = abcdef  k = 4  list: ['a', 'b', 'c', 'd', 'e', 'f']
  bước 1: hoán đổi chỉ số 0 và 3 -> dbcaef
  bước 2: hoán đổi chỉ số 1 và 2 -> dcbaef
  dừng với i=2, j=1
s = xyz  k = 3  list: ['x', 'y', 'z']
  bước 1: hoán đổi chỉ số 0 và 2 -> zyx
  dừng với i=1, j=1
s = hey  k = 1  list: ['h', 'e', 'y']
  dừng với i=0, j=0


In [15]:
def trace_stack(s, k):
    stack, out = [], []
    print("đẩy vào stack:")
    for c in s[:k]:
        stack.append(c)
        print(f"  push {c!r} -> stack={stack}")
    print("lấy ra:")
    while stack:
        c = stack.pop()
        out.append(c)
        print(f"  pop {c!r} -> out={out}")
    out.append(s[k:])
    return ''.join(out)

assert trace_stack("abcd", 2) == "bacd"

đẩy vào stack:
  push 'a' -> stack=['a']
  push 'b' -> stack=['a', 'b']
lấy ra:
  pop 'b' -> out=['b']
  pop 'a' -> out=['b', 'a']


## 9. Code nộp LeetCode

In [16]:
from typing import List, Optional

class Solution:
    def reversePrefix(self, s: str, k: int) -> str:
        return s[:k][::-1] + s[k:]

## 10. Vì sao thuật toán đúng

- **Chia đôi chính xác:** với `1 <= k <= n`, `s[:k] + s[k:] = s`, hai phần không giao nhau và phủ hết chuỗi. Đề chỉ đảo phần đầu `k` ký tự nên phần đuôi `s[k:]` được giữ nguyên.
- **Đảo đúng:** `s[:k][::-1]` đưa ký tự ở vị trí `i` (`0 <= i < k`) đến vị trí `k - 1 - i`, đúng định nghĩa "đảo ngược đoạn có độ dài `k`".
- **Hai con trỏ:** mỗi lần hoán đổi `(i, j)` với `i + j = k - 1` đặt hai ký tự đối xứng vào đúng chỗ; sau `⌊k / 2⌋` lần, mọi cặp được xử lý (ký tự giữa, nếu `k` lẻ, tự ở đúng chỗ).
- **Ngăn xếp:** LIFO đảo đúng thứ tự của `k` ký tự đã push.
- **Biên:** `k = 1` đoạn một ký tự đảo là chính nó; `k = n` thì đuôi rỗng và kết quả là `s[::-1]`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| Vòng lặp từng ký tự | `O(n)` (nối chuỗi có thể `O(n²)` lý thuyết) | `O(n)` |
| Slicing | `O(n)` | `O(n)` |
| Hai con trỏ | `O(n)` | `O(n)` |
| Ngăn xếp | `O(n)` | `O(n)` |
| Đệ quy | `O(k²)` | `O(k)` |

`n = len(s)`. Chuỗi bất biến nên mọi cách phải tạo ít nhất một chuỗi mới `O(n)`; không có cách `O(1)` bộ nhớ trong Python với `str`. Với `n <= 100` mọi cách đều tức thì.

## 12. Bộ test

Ví dụ của đề, các biên (`n = 1`, `k = 1`, `k = n`, `k = n - 1`, chuỗi toàn một ký tự, palindrome, `n = 100`) và so sánh **tất cả cách** trên input ngẫu nhiên hợp lệ.

In [17]:
impls = [SolutionLoop(), SolutionSlice(), SolutionTwoPointers(), SolutionStack(), SolutionRecursive(), Solution()]

cases = [
    (("abcd", 2), "bacd"), (("xyz", 3), "zyx"), (("hey", 1), "hey"),
    (("a", 1), "a"), (("ab", 2), "ba"), (("ab", 1), "ab"), (("abcde", 4), "dcbae"), (("abcde", 5), "edcba"),
    (("aaaa", 3), "aaaa"), (("abba", 4), "abba"), (("abba", 2), "baba"), (("z" * 100, 100), "z" * 100),
    (("abcdefghij" * 10, 100), ("abcdefghij" * 10)[::-1]), (("abcdefghij" * 10, 1), "abcdefghij" * 10),
]
for (s, k), exp in cases:
    assert 1 <= len(s) <= 100 and s.islower() and 1 <= k <= len(s)
    for impl in impls:
        got = impl.reversePrefix(s, k)
        assert got == exp, (type(impl).__name__, s, k, got, exp)
print("ví dụ + edge OK:", len(cases), "ca")

ví dụ + edge OK: 14 ca


In [18]:
import random, string
random.seed(37940)
for _ in range(1000):
    n = random.randint(1, 100)
    alpha = random.choice(['ab', 'abc', string.ascii_lowercase])
    s = ''.join(random.choice(alpha) for _ in range(n))
    k = random.randint(1, n)
    exp = ''.join(reversed(s[:k])) + s[k:]
    outs = [impl.reversePrefix(s, k) for impl in impls]
    assert all(o == exp for o in outs), (s, k, outs, exp)
    assert len(exp) == n and exp[k:] == s[k:]
print("1000 test ngẫu nhiên (seed=37940): 6 cách giống nhau")

1000 test ngẫu nhiên (seed=37940): 6 cách giống nhau


In [19]:
# Tính chất: áp dụng hai lần cùng k trả về chuỗi gốc (phép đảo là nghịch đảo của chính nó)
for _ in range(300):
    n = random.randint(1, 100)
    s = ''.join(random.choice(string.ascii_lowercase) for _ in range(n))
    k = random.randint(1, n)
    once = Solution().reversePrefix(s, k)
    assert Solution().reversePrefix(once, k) == s
    assert Solution().reversePrefix(s, n) == s[::-1]
print("tính chất đối hợp (involution) OK")

tính chất đối hợp (involution) OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: gán trực tiếp vào chuỗi

Chuỗi bất biến: `s[i] = c` gây `TypeError`. Phải chuyển sang list hoặc dựng chuỗi mới.

In [20]:
def wrong_assign(s, k):
    i, j = 0, k - 1
    while i < j:
        s[i], s[j] = s[j], s[i]                 # SAI: s là str
        i, j = i + 1, j - 1
    return s

try:
    wrong_assign("abcd", 2)
except TypeError as e:
    print("TypeError:", e)
assert Solution().reversePrefix("abcd", 2) == "bacd"

TypeError: 'str' object does not support item assignment


### Lỗi 2: nhầm slicing bước âm `s[:k:-1]`

`s[:k:-1]` đi từ cuối chuỗi về `k+1` (không gồm `k`), không phải "đảo `k` ký tự đầu".

In [21]:
def wrong_negative_slice(s, k):
    return s[:k:-1] + s[k:]                      # SAI: lấy ngược từ CUỐI chuỗi

print(wrong_negative_slice("abcd", 2), "!=", Solution().reversePrefix("abcd", 2))
assert wrong_negative_slice("abcd", 2) == "dcd" != "bacd"

dcd != bacd


In [22]:
print(repr("abcd"[:2:-1]), repr("abcd"[1::-1]))
assert "abcd"[:2:-1] == "d" and "abcd"[1::-1] == "ba"      # [:k:-1] sai; [k-1::-1] mới đúng

'd' 'ba'


### Lỗi 3: đảo cả chuỗi

Đọc vội đề rồi trả `s[::-1]`. Chỉ đúng khi `k = len(s)`.

In [23]:
def wrong_reverse_all(s, k):
    return s[::-1]                                 # SAI: đảo cả chuỗi

print(wrong_reverse_all("abcd", 2), "!=", Solution().reversePrefix("abcd", 2))
assert wrong_reverse_all("abcd", 2) == "dcba" != "bacd"
assert wrong_reverse_all("xyz", 3) == Solution().reversePrefix("xyz", 3)

dcba != bacd


### Lỗi 4: quên nối phần đuôi `s[k:]`

Chỉ trả về đoạn đầu đã đảo làm mất phần còn lại.

In [24]:
def wrong_forget_tail(s, k):
    return s[:k][::-1]                             # SAI: thiếu s[k:]

print(wrong_forget_tail("abcd", 2), "!=", Solution().reversePrefix("abcd", 2))
assert wrong_forget_tail("abcd", 2) == "ba" != "bacd"

ba != bacd


### Lỗi 5: lệch một đơn vị khi chọn `k` hoặc `k - 1`

Đảo `s[:k - 1]` hoặc dùng `j = k` thay vì `k - 1` trong hai con trỏ làm sai chỉ số.

In [25]:
def wrong_off_by_one(s, k):
    return s[:k - 1][::-1] + s[k - 1:]             # SAI: chỉ đảo k-1 ký tự

print(wrong_off_by_one("abcd", 3), "!=", Solution().reversePrefix("abcd", 3))
assert wrong_off_by_one("abcd", 3) == "bacd" != "cbad" == Solution().reversePrefix("abcd", 3)

bacd != cbad


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đảo một đoạn/tiền tố của chuỗi" -> `s[:k][::-1] + s[k:]`.
- Nhớ: **chuỗi bất biến**; muốn sửa tại chỗ thì `list(s)` rồi `''.join(...)`.
- `[::-1]` đảo cả chuỗi; đừng trộn bước âm với `start/stop` nếu chưa chắc (dùng `reversed` hoặc cắt trước rồi đảo).
- Hai con trỏ `i, j = 0, k - 1` là mẫu đảo tại chỗ chuẩn; dừng khi `i >= j`.
- Kiểm tra biên: `k = 1` (không đổi), `k = n` (đảo hết).
- Kết quả luôn là hoán vị của `s`, cùng độ dài và cùng đuôi `s[k:]` -> dùng làm assert tự kiểm tra.

## 15. Biến thể của bài

1. **Reverse String** (LeetCode 344): đảo cả mảng ký tự tại chỗ, `O(1)` bộ nhớ.
2. **Reverse String II** (541): cứ `2k` ký tự thì đảo `k` ký tự đầu của mỗi đoạn.
3. **Reverse Words in a String III** (557): đảo từng từ nhưng giữ thứ tự các từ.
4. **Reverse Only Letters** (917): chỉ đảo các chữ cái, giữ nguyên ký tự khác (xem bài 3823 cùng họ).
5. **Reverse Prefix of Word** (2000): đảo tiền tố đến lần xuất hiện đầu tiên của một ký tự cho trước.
6. **Rotate String** (796): kiểm tra một chuỗi có là phép xoay của chuỗi kia.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| Vòng lặp từng ký tự | `s[k - 1 - i]` cho `i < k` | `O(n)` | `O(n)` | Hiểu ánh xạ vị trí |
| Slicing | `s[:k][::-1] + s[k:]` | `O(n)` | `O(n)` | Nộp bài |
| Hai con trỏ | hoán đổi `i`, `j` trên list | `O(n)` | `O(n)` | Mẫu đảo tại chỗ (mảng) |
| Ngăn xếp | push `k` ký tự rồi pop | `O(n)` | `O(n)` | Liên hệ stack |
| Đệ quy | `rev(t[1:]) + t[0]` | `O(k²)` | `O(k)` | Học bài toán con |

Nộp cách slicing.

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().reversePrefix("leetcode", 4)` và `Solution().reversePrefix("python", 6)` ra gì?

**Bài 2 (tính tay):** với `s = "abcdefg"`, `k = 5`, viết đầu, đầu đã đảo, đuôi và kết quả.

**Bài 3 (điền TODO):** hoàn thành `reverse_suffix(s, k)` đảo `k` ký tự **cuối** của `s`.

**Bài 4 (tự viết lại):** viết `reverse_prefix_of_word(word, ch)` (LeetCode 2000): đảo tiền tố kết thúc tại lần xuất hiện **đầu tiên** của `ch`; nếu không có `ch` thì giữ nguyên.

**Bài 5 (mini challenge):** viết `reverse_every_k(s, k)` (LeetCode 541 dạng đơn giản): cứ mỗi đoạn `2k` ký tự, đảo `k` ký tự đầu của đoạn.

In [26]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().reversePrefix("leetcode", 4), Solution().reversePrefix("python", 6))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def reverse_suffix_todo(s, k):
    # TODO: tách s thành s[:-k] và s[-k:], chỉ đảo phần sau
    return s

teelcode nohtyp


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: `"leet"` đảo thành `"teel"`; `"python"` đảo cả chuỗi. Bài 2: đầu là `abcde`.

**Mức 2 (vừa):** Bài 3: `s[:len(s) - k] + s[len(s) - k:][::-1]`. Bài 4: dùng `word.find(ch)`; nếu `-1` thì trả `word`, ngược lại dùng `reversePrefix` với `k = idx + 1`. Bài 5: duyệt `i` theo bước `2k`, mỗi lần đảo `s[i:i+k]`.

**Mức 3 (gần đáp án):** Bài 1: `"teelcode"` và `"nohtyp"`. Bài 2: kết quả `"edcbafg"`. Bài 4: `reverse_prefix_of_word("abcdefd", "d") = "dcbaefd"`. Bài 5: `reverse_every_k("abcdefg", 2) = "bacdfeg"`.

## 19. Đáp án tham khảo cho bài tập

In [27]:
# Bài 1, 2
assert Solution().reversePrefix("leetcode", 4) == "teelcode" and Solution().reversePrefix("python", 6) == "nohtyp"
assert "abcdefg"[:5] == "abcde" and "abcdefg"[:5][::-1] == "edcba" and Solution().reversePrefix("abcdefg", 5) == "edcbafg"

# Bài 3
def reverse_suffix(s, k):
    cut = len(s) - k
    return s[:cut] + s[cut:][::-1]
assert reverse_suffix("abcd", 2) == "abdc" and reverse_suffix("xyz", 3) == "zyx" and reverse_suffix("hey", 1) == "hey"

# Bài 4
def reverse_prefix_of_word(word, ch):
    idx = word.find(ch)
    if idx == -1:
        return word
    return Solution().reversePrefix(word, idx + 1)
assert reverse_prefix_of_word("abcdefd", "d") == "dcbaefd"
assert reverse_prefix_of_word("xyxzxe", "z") == "zxyxxe" and reverse_prefix_of_word("abcd", "z") == "abcd"

# Bài 5
def reverse_every_k(s, k):
    out = []
    for i in range(0, len(s), 2 * k):
        out.append(s[i:i + k][::-1] + s[i + k:i + 2 * k])
    return ''.join(out)
assert reverse_every_k("abcdefg", 2) == "bacdfeg"
assert reverse_every_k("abcd", 2) == "bacd" == Solution().reversePrefix("abcd", 2)
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đảo tiền tố":**

```python
return s[:k][::-1] + s[k:]
# đảo tại chỗ: chars = list(s); i, j = 0, k - 1; while i < j: hoán đổi; i += 1; j -= 1
```

**Checklist trước khi submit:**

- [ ] Chỉ đảo `k` ký tự **đầu**, phần `s[k:]` giữ nguyên và được nối lại?
- [ ] Không gán trực tiếp `s[i] = ...` (chuỗi bất biến)?
- [ ] Không dùng `s[:k:-1]` nhầm bước âm; không lệch `k` / `k - 1`?
- [ ] Thử `k = 1` (không đổi) và `k = len(s)` (đảo hết)?
- [ ] Thử ba ví dụ của đề `("abcd",2)`, `("xyz",3)`, `("hey",1)`?
- [ ] Chữ ký đúng `reversePrefix(self, s: str, k: int) -> str`?